# Import Component Definitions
This example shows how to import component definitions. It includes

- Download an example board
- Create a config file with component RLC and solder ball information
- Apply the config file to the example board

## Perform imports and define constants

Perform required imports.

In [1]:
import json
import tempfile
import time
from pathlib import Path

import toml
from ansys.aedt.core.examples.downloads import download_file
from pyedb import Edb


Define constants.

In [2]:
AEDT_VERSION = "2026.1"

Download the example PCB data.

In [3]:
temp_folder = tempfile.TemporaryDirectory(suffix=".ansys")
file_edb = download_file(source="pyaedt/edb/ANSYS-HSD_V1.aedb", local_path=temp_folder.name)

## Load example layout.

In [4]:
edbapp = Edb(edbpath=file_edb, version=AEDT_VERSION)

C:\actions-runner\_work\pyaedt-examples\pyaedt-examples\.venv\lib\site-packages\pyedb\generic\design_types.py:374: UserWarning: You are using PyEDB with grpc, which is currently in beta. Some feature might be missing or not working as expected. Please report any issue you find to the PyEDB team.
  warnings.warn(GRPC_BETA_WARNING, UserWarning)


PyEDB INFO: Logger is initialized in EDB.


PyEDB INFO: legacy v0.74.0


PyEDB INFO: Python version 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]


PyEDB INFO: Using PyEDB with gRPC as Beta until ANSYS 2027R1 official release.


...

PyEDB INFO: Database ANSYS-HSD_V1.aedb Opened in 2026.1


PyEDB INFO: Cell main Opened


PyEDB INFO: Refreshing the Components dictionary.


PyEDB INFO: Builder was initialized.


PyEDB INFO: EDB initialized.


## Create a config file with component information

Keywords

- **reference_designator**. Reference designator of the component.
- **part_type**. Part type of the component. Supported types are 'resistor', 'inductor', 'capacitor', 'ic', 'io',
'other'.
- **enabled**. Only available for RLC components.
- **solder_ball_properties**.
  - **shape**. Supported types are 'cylinder', 'spheroid', 'none'.
  - **diameter**.
  - **mid_diameter**.
  - **height**.
  - **material**.
- **port_properties**.
  - **reference_offset**.
  - **reference_size_auto**.
  - **reference_size_x**.
  - **reference_size_y**.
- **pin_pair_model**. RLC network. Multiple pin pairs are supported.
  - **first_pin**. First pin of the pin pair.
  - **second_pin**. Second pin of the pin pair.
  - **is_parallel**.
  - **resistance**.
  - **resistance_enabled**.
  - **inductance**.
  - **inductance_enabled**.
  - **capacitance**.
  - **capacitance_enabled**.

In [5]:
cfg = dict()
cfg["components"] = [
    {
        "reference_designator": "U1",
        "part_type": "io",
        "solder_ball_properties": {
            "shape": "spheroid",
            "diameter": "244um",
            "mid_diameter": "400um",
            "height": "300um",
            "material": "air",
        },
        "port_properties": {
            "reference_offset": "0.1mm",
            "reference_size_auto": True,
            "reference_size_x": 0,
            "reference_size_y": 0,
        },
    },
    {
        "reference_designator": "C375",
        "enabled": False,
        "pin_pair_model": [
            {
                "first_pin": "1",
                "second_pin": "2",
                "is_parallel": False,
                "resistance": "10ohm",
                "resistance_enabled": True,
                "inductance": "1nH",
                "inductance_enabled": False,
                "capacitance": "10nF",
                "capacitance_enabled": True,
            }
        ],
    },
]

In [6]:
cfg_file_path = Path(temp_folder.name) / "cfg.json"
with open(cfg_file_path, "w") as f:
    json.dump(cfg, f, indent=4, ensure_ascii=False)

Equivalent toml file looks like below

In [7]:
toml_string = toml.dumps(cfg)
print(toml_string)

[[components]]
reference_designator = "U1"
part_type = "io"

[components.solder_ball_properties]
shape = "spheroid"
diameter = "244um"
mid_diameter = "400um"
height = "300um"
material = "air"
[components.port_properties]
reference_offset = "0.1mm"
reference_size_auto = true
reference_size_x = 0
reference_size_y = 0
[[components]]
reference_designator = "C375"
enabled = false
[[components.pin_pair_model]]
first_pin = "1"
second_pin = "2"
is_parallel = false
resistance = "10ohm"
resistance_enabled = true
inductance = "1nH"
inductance_enabled = false
capacitance = "10nF"
capacitance_enabled = true





## Apply config file

In [8]:
edbapp.configuration.load(cfg_file_path, apply_file=True)

PyEDB INFO: Updating nets finished. Time lapse 0:00:00


PyEDB INFO: Updating components finished. Time lapse 0:00:00.047629


PyEDB INFO: Creating pin groups finished. Time lapse 0:00:00


PyEDB INFO: Placing sources finished. Time lapse 0:00:00


PyEDB INFO: Applying materials finished. Time lapse 0:00:00


...

PyEDB INFO: Placing ports finished. Time lapse 0:00:00


PyEDB INFO: Placing terminals completed in 0.0000 seconds.


PyEDB INFO: Placing probes finished. Time lapse 0:00:00


PyEDB INFO: Applying operations completed in 0.0000 seconds.


PyEDB INFO: Applying setups completed in 0.0000 seconds.


Save and close Edb

The temporary folder will be deleted once the execution of this script is finished. Replace **edbapp.save()** with
**edbapp.save_as("C:/example.aedb")** to keep the example project.

In [9]:
edbapp.save()
edbapp.close()

PyEDB INFO: RPC session released (open databases: 0)


True

In [10]:
# Wait 3 seconds before cleaning the temporary directory.
time.sleep(3)

### Clean up

All project files are saved in the folder ``temp_folder.name``.
If you've run this example as a Jupyter notebook, you
can retrieve those project files. The following cell
removes all temporary files, including the project folder.

In [11]:
temp_folder.cleanup()